# 🥔 YOLOv8 - Détection de Maladies des Pommes de Terre
### Google Colab | GPU T4 | Dataset: Early Blight / Late Blight / Healthy
---

## ✅ ÉTAPE 1 — Vérifier le GPU

In [ ]:
# Vérifier que le GPU T4 est bien activé
!nvidia-smi
import torch
print(f'\n✅ CUDA disponible : {torch.cuda.is_available()}')
print(f'✅ GPU : {torch.cuda.get_device_name(0)}')
print(f'✅ VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## ✅ ÉTAPE 2 — Installer YOLOv8

In [ ]:
!pip install ultralytics -q
!pip install matplotlib scikit-learn pyyaml -q

from ultralytics import YOLO
import ultralytics
ultralytics.checks()
print('\n✅ YOLOv8 prêt !')

## ✅ ÉTAPE 3 — Préparer le Dataset (conversion vers format YOLO)

In [ ]:
import os, shutil, random, yaml
from pathlib import Path
from sklearn.model_selection import train_test_split

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 📁 CHEMIN VERS TON DATASET (déjà dans Colab)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
DATASET_ROOT = '/content'   # Les dossiers Potato__ sont ici
OUTPUT_DIR   = '/content/potato_yolo'

# Détection automatique des classes
CLASSES = sorted([
    d for d in os.listdir(DATASET_ROOT)
    if os.path.isdir(os.path.join(DATASET_ROOT, d))
    and 'potato' in d.lower() or 'Potato' in d
])

# Si la détection automatique ne fonctionne pas, décommente ces lignes :
# CLASSES = ['Potato__Early_blight', 'Potato__Late_blight', 'Potato__healthy']

print(f'📂 Classes trouvées ({len(CLASSES)}) :')
for i, cls in enumerate(CLASSES):
    imgs = list(Path(DATASET_ROOT, cls).glob('*.*'))
    imgs = [f for f in imgs if f.suffix.lower() in ['.jpg','.jpeg','.png']]
    print(f'   [{i}] {cls} → {len(imgs)} images')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Conversion vers format YOLO
# Dataset de classification → Object Detection
# BBox = 90% de la surface de l'image
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

random.seed(42)

# Créer la structure de dossiers YOLO
for split in ['train', 'val', 'test']:
    Path(OUTPUT_DIR, 'images', split).mkdir(parents=True, exist_ok=True)
    Path(OUTPUT_DIR, 'labels', split).mkdir(parents=True, exist_ok=True)

stats = {'train': 0, 'val': 0, 'test': 0}

for class_id, class_name in enumerate(CLASSES):
    class_path = Path(DATASET_ROOT, class_name)
    images = [f for f in class_path.glob('*.*')
              if f.suffix.lower() in ['.jpg', '.jpeg', '.png']]

    if not images:
        print(f'⚠️  Aucune image dans : {class_name}')
        continue

    # Split 70% / 20% / 10%
    train_imgs, temp = train_test_split(images, test_size=0.3, random_state=42)
    val_imgs, test_imgs = train_test_split(temp, test_size=0.33, random_state=42)

    for split_name, split_imgs in [('train', train_imgs), ('val', val_imgs), ('test', test_imgs)]:
        for img_path in split_imgs:
            # Nom unique pour éviter les conflits
            new_name = f'{class_name}_{img_path.name}'

            # Copier l'image
            dst_img = Path(OUTPUT_DIR, 'images', split_name, new_name)
            shutil.copy2(img_path, dst_img)

            # Créer le label YOLO (bbox couvre 90% de l'image, centré)
            label_name = Path(new_name).stem + '.txt'
            dst_label = Path(OUTPUT_DIR, 'labels', split_name, label_name)
            with open(dst_label, 'w') as f:
                # format: class_id x_center y_center width height (normalisé 0-1)
                f.write(f'{class_id} 0.5 0.5 0.9 0.9\n')

            stats[split_name] += 1

# Créer data.yaml
# Noms affichés plus propres
clean_names = []
for cls in CLASSES:
    name = cls.replace('Potato__', '').replace('_', ' ')
    clean_names.append(name)

data_yaml = {
    'path': OUTPUT_DIR,
    'train': 'images/train',
    'val':   'images/val',
    'test':  'images/test',
    'nc':    len(CLASSES),
    'names': clean_names
}

YAML_PATH = f'{OUTPUT_DIR}/data.yaml'
with open(YAML_PATH, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print('\n✅ Conversion terminée !')
print(f'   Train : {stats["train"]} images')
print(f'   Val   : {stats["val"]} images')
print(f'   Test  : {stats["test"]} images')
print(f'   Classes : {clean_names}')
print(f'   data.yaml : {YAML_PATH}')

## ✅ ÉTAPE 4 — Visualiser quelques images du dataset

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2, random
from pathlib import Path

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Exemples du Dataset - Maladies Pommes de Terre', fontsize=16, fontweight='bold')

colors = ['#e74c3c', '#e67e22', '#27ae60']  # rouge, orange, vert

train_imgs = list(Path(OUTPUT_DIR, 'images', 'train').glob('*.jpg')) + \
             list(Path(OUTPUT_DIR, 'images', 'train').glob('*.jpeg')) + \
             list(Path(OUTPUT_DIR, 'images', 'train').glob('*.png'))

samples = random.sample(train_imgs, min(6, len(train_imgs)))

for ax, img_path in zip(axes.flat, samples):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Trouver la classe à partir du nom du fichier
    cls_name = 'Inconnu'
    color = 'gray'
    for i, cls in enumerate(CLASSES):
        if cls.lower() in img_path.name.lower():
            cls_name = clean_names[i]
            color = colors[i % len(colors)]
            break

    ax.imshow(img)
    h, w = img.shape[:2]
    rect = mpatches.Rectangle(
        (w*0.05, h*0.05), w*0.9, h*0.9,
        linewidth=3, edgecolor=color, facecolor='none'
    )
    ax.add_patch(rect)
    ax.set_title(cls_name, fontsize=12, color=color, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig('/content/dataset_preview.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Aperçu sauvegardé : /content/dataset_preview.png')

## ✅ ÉTAPE 5 — Entraîner YOLOv8 🚀

In [ ]:
from ultralytics import YOLO

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ⚙️ PARAMÈTRES D'ENTRAÎNEMENT
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
MODEL_SIZE  = 's'    # 'n'=nano (rapide) | 's'=small (recommandé T4) | 'm'=medium
EPOCHS      = 100    # 100 epochs minimum pour de bons résultats
IMG_SIZE    = 640    # T4 supporte 640 facilement
BATCH_SIZE  = 32     # T4 a 15GB VRAM → batch 32 est optimal
PROJECT     = '/content/potato_runs'
RUN_NAME    = f'yolov8{MODEL_SIZE}_potato'
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Charger le modèle pré-entraîné (Transfer Learning depuis COCO)
model = YOLO(f'yolov8{MODEL_SIZE}.pt')

print(f'🚀 Démarrage entraînement YOLOv8{MODEL_SIZE}...')
print(f'   Epochs     : {EPOCHS}')
print(f'   Image Size : {IMG_SIZE}')
print(f'   Batch      : {BATCH_SIZE}')
print(f'   Classes    : {clean_names}')
print(f'   GPU        : T4 ✅')
print('─' * 50)

results = model.train(
    data=YAML_PATH,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project=PROJECT,
    name=RUN_NAME,

    # ── Optimiseur ──
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,

    # ── Augmentations spéciales pour feuilles de plantes ──
    hsv_h=0.02,       # légère variation de teinte
    hsv_s=0.9,        # forte saturation (couleur des taches)
    hsv_v=0.5,        # variation de luminosité
    degrees=45.0,     # rotation (feuilles dans tous les sens)
    translate=0.1,
    scale=0.6,
    shear=5.0,
    perspective=0.0005,
    flipud=0.5,
    fliplr=0.5,
    mosaic=1.0,       # mélange 4 images
    mixup=0.15,       # mélange 2 images
    copy_paste=0.1,

    # ── Divers ──
    patience=25,      # early stopping
    save=True,
    save_period=20,
    plots=True,
    verbose=True,
    device=0,         # GPU T4
    exist_ok=True,
    workers=4,
    amp=True,         # Mixed Precision → plus rapide sur T4
)

BEST_MODEL = f'{PROJECT}/{RUN_NAME}/weights/best.pt'
print(f'\n✅ Entraînement terminé !')
print(f'   Meilleur modèle : {BEST_MODEL}')

## ✅ ÉTAPE 6 — Évaluer le modèle

In [ ]:
from ultralytics import YOLO

model = YOLO(BEST_MODEL)

metrics = model.val(
    data=YAML_PATH,
    split='test',
    plots=True,
    verbose=True
)

print('\n' + '='*50)
print('📊 RÉSULTATS SUR LE JEU DE TEST')
print('='*50)
print(f'  mAP@50     : {metrics.box.map50:.4f}  ({metrics.box.map50*100:.1f}%)')
print(f'  mAP@50-95  : {metrics.box.map:.4f}  ({metrics.box.map*100:.1f}%)')
print(f'  Précision  : {metrics.box.mp:.4f}  ({metrics.box.mp*100:.1f}%)')
print(f'  Rappel     : {metrics.box.mr:.4f}  ({metrics.box.mr*100:.1f}%)')

# Évaluation qualitative
mAP = metrics.box.map50
if mAP >= 0.85:
    print(f'\n🏆 Excellent résultat ! Le modèle est prêt pour la production.')
elif mAP >= 0.70:
    print(f'\n✅ Bon résultat. Tu peux augmenter les epochs ou passer à yolov8m.')
elif mAP >= 0.50:
    print(f'\n⚠️  Résultat moyen. Essaie plus d\'epochs ou annote les bbox manuellement.')
else:
    print(f'\n❌ Résultat faible. Vérifier le dataset ou annoter manuellement les images.')

## ✅ ÉTAPE 7 — Courbes d'entraînement

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

results_csv = f'{PROJECT}/{RUN_NAME}/results.csv'
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('📈 Courbes d\'Entraînement YOLOv8 - Pommes de Terre',
             fontsize=15, fontweight='bold')

plots_config = [
    ('train/box_loss',      'Loss Box (Train)',    '#e74c3c'),
    ('train/cls_loss',      'Loss Classe (Train)', '#e67e22'),
    ('val/box_loss',        'Loss Box (Val)',       '#3498db'),
    ('val/cls_loss',        'Loss Classe (Val)',    '#9b59b6'),
    ('metrics/mAP50(B)',    'mAP@50',              '#27ae60'),
    ('metrics/mAP50-95(B)', 'mAP@50-95',           '#1abc9c'),
]

for ax, (col, title, color) in zip(axes.flat, plots_config):
    if col in df.columns:
        ax.plot(df['epoch'], df[col], color=color, linewidth=2.5)
        ax.fill_between(df['epoch'], df[col], alpha=0.1, color=color)
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xlabel('Epoch', fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.set_facecolor('#f8f9fa')
    else:
        ax.set_title(f'{title} (non disponible)', fontsize=10, color='gray')
        ax.axis('off')

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Courbes sauvegardées : /content/training_curves.png')

## ✅ ÉTAPE 8 — Tester sur des images

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2, random
from pathlib import Path
from ultralytics import YOLO

model = YOLO(BEST_MODEL)

# Prendre 6 images de test aléatoires
test_imgs = list(Path(OUTPUT_DIR, 'images', 'test').glob('*.*'))
test_imgs = [f for f in test_imgs if f.suffix.lower() in ['.jpg','.jpeg','.png']]
samples   = random.sample(test_imgs, min(6, len(test_imgs)))

colors_map = {'Early blight': '#e74c3c', 'Late blight': '#e67e22', 'healthy': '#27ae60'}
default_color = '#3498db'

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('🔍 Prédictions YOLOv8 - Test', fontsize=15, fontweight='bold')

for ax, img_path in zip(axes.flat, samples):
    results = model.predict(str(img_path), conf=0.4, verbose=False)
    result  = results[0]

    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    h, w = img.shape[:2]

    if result.boxes and len(result.boxes) > 0:
        for box in result.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            conf    = box.conf[0].item()
            cls_id  = int(box.cls[0].item())
            cls_name = clean_names[cls_id] if cls_id < len(clean_names) else '?'
            color   = colors_map.get(cls_name, default_color)

            rect = mpatches.Rectangle(
                (x1, y1), x2-x1, y2-y1,
                linewidth=3, edgecolor=color, facecolor='none'
            )
            ax.add_patch(rect)
            ax.text(x1, y1-6, f'{cls_name} {conf:.0%}',
                    color='white', fontsize=9, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2', fc=color, alpha=0.85))
        ax.set_title(cls_name, fontsize=11, color=color, fontweight='bold')
    else:
        ax.set_title('Aucune détection', fontsize=10, color='gray')

    ax.axis('off')

plt.tight_layout()
plt.savefig('/content/predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Prédictions sauvegardées : /content/predictions.png')

## ✅ ÉTAPE 9 — Exporter et télécharger le modèle

In [ ]:
from ultralytics import YOLO
from google.colab import files
import shutil

model = YOLO(BEST_MODEL)

# ── Export ONNX (universel) ──
print('📤 Export ONNX...')
onnx_path = model.export(format='onnx', imgsz=640)
print(f'   ✅ ONNX : {onnx_path}')

# ── Zipper tous les résultats ──
print('\n📦 Création du ZIP avec tous les résultats...')
zip_path = '/content/potato_yolo_results'
shutil.make_archive(zip_path, 'zip', f'{PROJECT}/{RUN_NAME}')
print(f'   ✅ ZIP créé : {zip_path}.zip')

# ── Télécharger ──
print('\n⬇️  Téléchargement...')
files.download(BEST_MODEL)           # best.pt
files.download(f'{zip_path}.zip')    # tous les résultats
print('✅ Téléchargement lancé !')

## ✅ ÉTAPE 10 — Résumé final

In [ ]:
print('\n' + '🥔 ' * 20)
print('\n  ✅ PIPELINE COMPLET TERMINÉ !')
print('\n' + '═' * 50)
print(f'  Modèle       : YOLOv8{MODEL_SIZE} (Transfer Learning)')
print(f'  Classes      : {clean_names}')
print(f'  Epochs       : {EPOCHS}')
print(f'  GPU          : T4 (Google Colab)')
print(f'  mAP@50       : {metrics.box.map50*100:.1f}%')
print('\n  📁 Fichiers importants :')
print(f'     best.pt   → {BEST_MODEL}')
print(f'     best.onnx → {onnx_path}')
print(f'     results   → {PROJECT}/{RUN_NAME}/')
print('═' * 50)

print('\n  💡 Prochaines étapes pour améliorer :')
print('     1. Annoter manuellement les bbox avec LabelImg')
print('        → résultats beaucoup plus précis')
print('     2. Passer à yolov8m si GPU T4 disponible')
print('     3. Augmenter le dataset avec plus d\'images')
print('     4. Intégrer avec SORT pour le tracking vidéo')
print('\n' + '🥔 ' * 20)

## ✅ ÉTAPE 3 — Préparer le Dataset (depuis Label Studio)

Cette section adapte la préparation du dataset pour utiliser un dataset déjà annoté et exporté au format YOLO depuis Label Studio. Vous devrez télécharger le fichier ZIP exporté par Label Studio.

In [ ]:
import os
from google.colab import files
import shutil
import yaml

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ⬆️ UPLOAD ET DÉCOMPRESSION DU DATASET DE LABEL STUDIO
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("Veuillez uploader le fichier ZIP de votre dataset exporté depuis Label Studio.")
uploaded = files.upload()

# Trouver le nom du fichier ZIP uploadé
zip_filename = next(iter(uploaded))

# Décompresser le fichier
OUTPUT_DIR = '/content/potato_yolo_label_studio_data'
shutil.unpack_archive(zip_filename, OUTPUT_DIR)

print(f'✅ Dataset décompressé dans : {OUTPUT_DIR}')

# Vérifier la structure et trouver le data.yaml
LABEL_STUDIO_YAML_PATH = os.path.join(OUTPUT_DIR, 'data.yaml')
if not os.path.exists(LABEL_STUDIO_YAML_PATH):
    # Si data.yaml n'est pas directement à la racine, chercher un répertoire enfant (ex: 'dataset')
    subdirs = [d for d in os.listdir(OUTPUT_DIR) if os.path.isdir(os.path.join(OUTPUT_DIR, d))]
    for subdir in subdirs:
        if os.path.exists(os.path.join(OUTPUT_DIR, subdir, 'data.yaml')):
            OUTPUT_DIR = os.path.join(OUTPUT_DIR, subdir)
            LABEL_STUDIO_YAML_PATH = os.path.join(OUTPUT_DIR, 'data.yaml')
            break

if not os.path.exists(LABEL_STUDIO_YAML_PATH):
    raise FileNotFoundError(f"Le fichier data.yaml n'a pas été trouvé dans {OUTPUT_DIR} ou ses sous-répertoires.")

YAML_PATH = LABEL_STUDIO_YAML_PATH
print(f'✅ Chemin du fichier data.yaml : {YAML_PATH}')

# Charger le YAML pour obtenir les noms de classes
with open(YAML_PATH, 'r') as f:
    data_yaml_content = yaml.safe_load(f)

clean_names = data_yaml_content.get('names', [])
num_classes = data_yaml_content.get('nc', 0)

print(f'📂 Classes détectées ({num_classes}) : {clean_names}')

# Compter les images (simplifié pour l'exemple, Label Studio fait déjà les splits)
stats = {'train': 0, 'val': 0, 'test': 0}
for split in ['train', 'val', 'test']:
    split_path = os.path.join(OUTPUT_DIR, 'images', split)
    if os.path.exists(split_path):
        stats[split] = len([f for f in os.listdir(split_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

print(f'   Train : {stats["train"]} images')
print(f'   Val   : {stats["val"]} images')
print(f'   Test  : {stats["test"]} images')


Veuillez uploader le fichier ZIP de votre dataset exporté depuis Label Studio.
